In [1]:
import warnings
warnings.filterwarnings('ignore')

import os
import itertools

import pandas as pd
import umap
import numpy as np
import joblib

from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_selection import VarianceThreshold
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score

import plotly.graph_objects as go
import plotly.express as px
from plotly.offline import plot


# Análisis de Clustering con UMAP + Gower + DBSCAN

---

## 📌 Descripción General
Este proyecto implementa un pipeline de análisis de clustering en 3D diseñado para datos mixtos (numéricos y categóricos). Combina:

- **Métrica de Gower**  
  Para calcular distancias en datos heterogéneos.

- **UMAP**  
  Reducción dimensional no lineal preservando relaciones locales y globales.

- **DBSCAN**  
  Algoritmo de clustering basado en densidad, robusto a ruido y formas irregulares.

---

## 🔍 Flujo de Trabajo

1. **Preprocesamiento**  
   - **Normalización**: Escala características al rango `[0, 1]`.  
   - **Filtrado**: Elimina variables con varianza cero (sin información útil).

2. **Cálculo de Distancias (Gower)**  
   - Construye una matriz de similitud adaptada para datos mixtos.  
   - Considera tanto atributos numéricos como categóricos en una misma medida.

3. **Reducción Dimensional (UMAP)**  
   - Optimización automática de hiperparámetros:  
     ```yaml
     n_neighbors: 3 – 100
     min_dist:    0.01 – 10
     spread:      0.3 – 15
     ```  
   - Genera un embedding 3D para visualización y clustering.

4. **Clustering (DBSCAN)**  
   - Búsqueda de parámetros óptimos:  
     ```yaml
     eps:         0.01 – 10
     min_samples: 3 – 100
     ```  
   - Detección automática del número de clusters.  
   - Identifica ruido (puntos no asignados a ningún cluster).

5. **Evaluación y Visualización**  
   - **Silhouette Score**: Mide cohesión y separación de clusters (rango: -1 a 1).  
   - **Visualización 3D interactiva**:  
     - Cada punto representa una observación.  
     - Colores distinguen clusters.  
     - Leyenda interactiva y ejes etiquetados.

---

## 🏆 Resultados Óptimos

### UMAP
```yaml
n_neighbors: 50
min_dist:    0.01
spread:      0.5
```
### DBSCAN
eps=10,min_samples=3

### Silhouette
0.913


In [2]:
# 1) Carga y definición de X
path = "data/lista_victima.csv"
df   = pd.read_csv(path)
X    = df.values

# 2) Escalado a [0,1]
scaler      = MinMaxScaler(feature_range=(0, 1))
X_scaled    = scaler.fit_transform(X)

# 3) Eliminar dimensiones de varianza cero
vt       = VarianceThreshold(threshold=0.0)
X_bin    = vt.fit_transform(X_scaled)


In [3]:
import gower

# Calcula la matriz de distancia Gower (n_samples × n_samples)
D_gower = gower.gower_matrix(X_bin)

In [4]:
# Parámetros de grid (igual que antes)
param_grid = {
    'n_neighbors': [3, 5, 15, 50, 100],
    'min_dist':    [0.01, 0.1, 0.5, 2, 10],
    'spread':      [0.3, 0.5, 2, 5, 15],
    # Fíjate: obligamos a usar precomputed
    'metric':      ['precomputed']
}
n_components = 3  # embedding 3D

# DBSCAN grid igual
dbscan_eps_list         = param_grid['min_dist']
dbscan_min_samples_list = param_grid['n_neighbors']

# Carpeta de salida
output_dir = "umap_victima_dbscan_html"
os.makedirs(output_dir, exist_ok=True)

i = 0
for n_n, m_d, sp, metric in itertools.product(
        param_grid['n_neighbors'],
        param_grid['min_dist'],
        param_grid['spread'],
        param_grid['metric']
    ):
    if m_d > sp:
        continue
    i += 1

    # 5.1) UMAP con precomputed
    reducer = umap.UMAP(
        n_neighbors = n_n,
        min_dist     = m_d,
        spread       = sp,
        n_components = n_components,
        metric       = 'precomputed',  # aquí
        random_state = 42
    )
    embedding = reducer.fit_transform(D_gower)   # PASAMOS LA MATRIZ D_gower

    # 5.2) Grid-search DBSCAN
    best_score  = -1
    best_labels = None
    best_params = {}
    for eps in dbscan_eps_list:
        for min_s in dbscan_min_samples_list:
            db     = DBSCAN(eps=eps, min_samples=min_s)
            labels = db.fit_predict(embedding)
            if len(set(labels)) > 1:
                score = silhouette_score(embedding, labels)
            else:
                score = -1
            if score > best_score:
                best_score  = score
                best_labels = labels
                best_params = {'eps': eps, 'min_samples': min_s}

    # 5.3) Guardar resultados
    df_out = pd.DataFrame({
        'umap_1': embedding[:, 0],
        'umap_2': embedding[:, 1],
        'umap_3': embedding[:, 2],
        'cluster': best_labels
    })
    df_out.to_csv(os.path.join(output_dir, f"comb_{i}_results.csv"),
                  index=False)

    # 5.4) Guardar modelos
    joblib.dump(reducer,
                os.path.join(output_dir, f"comb_{i}_umap_model.pkl"))
    joblib.dump(DBSCAN(**best_params).fit(embedding),
                os.path.join(output_dir, f"comb_{i}_dbscan_model.pkl"))

    # 5.5) Visualización 3D interactiva (igual que antes)
    fig = go.Figure()
    palette = px.colors.qualitative.Set1
    for idx, label in enumerate(np.unique(best_labels)):
        mask = (best_labels == label)
        name = 'Noise' if label == -1 else f'Cluster {label}'
        fig.add_trace(go.Scatter3d(
            x=embedding[mask, 0],
            y=embedding[mask, 1],
            z=embedding[mask, 2],
            mode='markers',
            marker=dict(size=3, opacity=0.7,
                        color=palette[idx % len(palette)]),
            name=name
        ))

    fig.update_layout(
        title=(
            f"UMAP-Gower (n_n={n_n}, min_d={m_d}, spread={sp})<br>"
            f"DBSCAN eps={best_params['eps']}, "
            f"min_samples={best_params['min_samples']} "
            f"(silhouette={best_score:.3f})"
        ),
        scene=dict(
            xaxis_title='UMAP 1',
            yaxis_title='UMAP 2',
            zaxis_title='UMAP 3'
        ),
        legend=dict(itemsizing='constant',
                    bgcolor='rgba(0,0,0,0)', borderwidth=1),
        margin=dict(l=0, r=0, b=0, t=80)
    )
    plot(fig,
         filename=os.path.join(output_dir, f"comb_{i}_dbscan.html"),
         auto_open=False)

    print(f"[Iter {i}] UMAP-Gower: n_n={n_n}, min_d={m_d}, spread={sp} "
          f"-> DBSCAN eps={best_params['eps']}, "
          f"min_s={best_params['min_samples']} "
          f"(silhouette={best_score:.3f})")

print(f"Total iteraciones: {i}. Revisa '{output_dir}/'")

[Iter 1] UMAP-Gower: n_n=3, min_d=0.01, spread=0.3 -> DBSCAN eps=0.5, min_s=5 (silhouette=0.504)
[Iter 2] UMAP-Gower: n_n=3, min_d=0.01, spread=0.5 -> DBSCAN eps=0.5, min_s=5 (silhouette=0.519)
[Iter 3] UMAP-Gower: n_n=3, min_d=0.01, spread=2 -> DBSCAN eps=2, min_s=5 (silhouette=0.703)
[Iter 4] UMAP-Gower: n_n=3, min_d=0.01, spread=5 -> DBSCAN eps=2, min_s=5 (silhouette=0.627)
[Iter 5] UMAP-Gower: n_n=3, min_d=0.01, spread=15 -> DBSCAN eps=10, min_s=15 (silhouette=0.461)
[Iter 6] UMAP-Gower: n_n=3, min_d=0.1, spread=0.3 -> DBSCAN eps=0.1, min_s=3 (silhouette=0.510)
[Iter 7] UMAP-Gower: n_n=3, min_d=0.1, spread=0.5 -> DBSCAN eps=2, min_s=5 (silhouette=0.533)
[Iter 8] UMAP-Gower: n_n=3, min_d=0.1, spread=2 -> DBSCAN eps=0.5, min_s=5 (silhouette=0.702)
[Iter 9] UMAP-Gower: n_n=3, min_d=0.1, spread=5 -> DBSCAN eps=2, min_s=5 (silhouette=0.708)
[Iter 10] UMAP-Gower: n_n=3, min_d=0.1, spread=15 -> DBSCAN eps=10, min_s=15 (silhouette=0.453)
[Iter 11] UMAP-Gower: n_n=3, min_d=0.5, spread=0.5 -

In [5]:
import os
import pandas as pd

def summarize_csv_clusters(directory: str) -> pd.DataFrame:
    """
    Recorre todos los archivos .csv en el directorio dado y devuelve un DataFrame
    con, para cada archivo:
      - file: nombre del archivo
      - count_minus_one: cantidad de valores -1 en la columna 'cluster'
      - num_categories: número de categorías únicas en la columna 'cluster'
      - error (opcional): mensaje de error si no se pudo procesar el archivo
    """
    records = []
    for filename in os.listdir(directory):
        if filename.lower().endswith(".csv"):
            path = os.path.join(directory, filename)
            try:
                df = pd.read_csv(path)
                if 'cluster' in df.columns:
                    count_minus_one = (df['cluster'] == -1).sum()
                    num_categories = df['cluster'].nunique()
                else:
                    count_minus_one = None
                    num_categories = None
                records.append({
                    'file': filename,
                    'count_minus_one': count_minus_one,
                    'num_categories': num_categories
                })
            except Exception as e:
                records.append({
                    'file': filename,
                    'count_minus_one': None,
                    'num_categories': None,
                    'error': str(e)
                })
    return pd.DataFrame(records)

# Ejemplo de uso:
path = r"umap_victima_dbscan_html"
summary_df = summarize_csv_clusters(path)
summary_df

,file,count_minus_one,num_categories
0,comb_10_results.csv,162,21
1,comb_11_results.csv,52,75
2,comb_12_results.csv,50,62
3,comb_13_results.csv,9,99
4,comb_14_results.csv,148,22
...,...,...,...
85,comb_88_results.csv,0,12
86,comb_89_results.csv,351,14
87,comb_8_results.csv,96,100
88,comb_90_results.csv,69,19


In [6]:
import re 

def load_iter_silhouette(filepath: str) -> pd.DataFrame:
    """
    Carga un archivo de texto donde cada línea tiene el formato:
      [Iter i] ... (silhouette=x.y)
    y devuelve un DataFrame con dos columnas:
      - iter: el número i
      - silhouette: el valor x.y
    """
    # Expresión regular para capturar el número de iter y el valor de silhouette
    pattern = re.compile(r'\[Iter\s+(\d+)\].*?\(silhouette=([0-9]*\.?[0-9]+)\)')
    
    registros = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for línea in f:
            m = pattern.search(línea)
            if m:
                iter_num = int(m.group(1))
                sil = float(m.group(2))
                registros.append({'iter': iter_num, 'silhouette': sil})
    
    # Construimos el DataFrame
    df = pd.DataFrame(registros, columns=['iter', 'silhouette'])
    return df

In [7]:
df_silhouette = load_iter_silhouette('cluster_victima.txt')
victima_cluster_metrics = summary_df.join(df_silhouette).drop(columns="iter")
victima_cluster_metrics.to_csv(r"data/metric_umap_DBSCAN_victima.csv",index=False)

In [10]:
victima_cluster_metrics.iloc[victima_cluster_metrics["silhouette"].idxmax()]

file               comb_60_results.csv
count_minus_one                      0
num_categories                       8
silhouette                       0.913
Name: 55, dtype: object

In [ ]:
victima_cluster_metrics.sort_values("silhouette", ascending=False).head(10)

,file,count_minus_one,num_categories,silhouette
55,comb_60_results.csv,0,8,0.913
73,comb_77_results.csv,299,14,0.907
54,comb_5_results.csv,159,21,0.905
72,comb_76_results.csv,0,14,0.899
74,comb_78_results.csv,3,11,0.892
79,comb_82_results.csv,360,13,0.889
56,comb_61_results.csv,0,12,0.879
65,comb_6_results.csv,56,228,0.869
78,comb_81_results.csv,0,13,0.868
57,comb_62_results.csv,0,11,0.856
